# C&W — Towards Evaluating the Robustness of Neural Networks (2017)

_Papers · Meta-learning, Bayesian & Robustness_

**A gradient-based attack that finds tiny, almost-invisible perturbations which fool any target class at 100% success.**

---

This notebook is a practice scaffold for the **C&W — Towards Evaluating the Robustness of Neural Networks (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
torch.manual_seed(0); np.random.seed(0)

# --- 0. Worked example: the margin loss f on a tiny logit vector. ---
Z = torch.tensor([2.0, 0.5, -1.0]); t = 2; kappa = 0.0
others = torch.cat([Z[:t], Z[t+1:]])                  # logits i != t
f = max((others.max() - Z[t]).item(), -kappa)
print("worked: Z=%s t=%d  max_{i!=t}Z=%.1f  Z_t=%.1f  f=%.1f"
      % (Z.tolist(), t, others.max().item(), Z[t].item(), f))   # -> f = 3.0
Zb = torch.tensor([2.0, 0.5, 3.0]); tb = 2                      # target winning, kappa=1
ob = torch.cat([Zb[:tb], Zb[tb+1:]])
print("worked2: raw margin=%.1f  f(kappa=1)=%.1f"
      % (ob.max().item()-Zb[tb].item(),
         max(ob.max().item()-Zb[tb].item(), -1.0)))             # -> f = -1.0

# --- 1. A tiny classifier on toy data living in [0,1]^2 (so the box constraint matters). ---
def make_data(n):
    X, y, centers = [], [], [(-1.5,-1.5),(1.5,-1.5),(0.0,1.6)]
    for c,(cx,cy) in enumerate(centers):
        X.append(np.random.randn(n,2)*0.45 + np.array([cx,cy])); y += [c]*n
    X = np.vstack(X).astype(np.float32); X = (X-X.min(0))/(X.max(0)-X.min(0))   # -> [0,1]
    return torch.tensor(X), torch.tensor(y)

Xtr, ytr = make_data(120)
clf = nn.Sequential(nn.Linear(2,32), nn.ReLU(), nn.Linear(32,32), nn.ReLU(), nn.Linear(32,3))
opt = torch.optim.Adam(clf.parameters(), lr=0.01)
for _ in range(400):
    opt.zero_grad(); F.cross_entropy(clf(Xtr), ytr).backward(); opt.step()
def Z(x): return clf(x)                                  # logits = output BEFORE softmax
print("\nclassifier train acc = %.3f" % (clf(Xtr).argmax(1)==ytr).float().mean().item())

# Correctly-classified points; target class = next class (a fixed mislabel).
idx = (clf(Xtr).argmax(1)==ytr).nonzero().squeeze()[:60]
Xc, yc = Xtr[idx].clone(), ytr[idx].clone(); tgt = (yc+1) % 3

# --- 2. The C&W L2 attack, built by hand: tanh box-constraint + margin loss + Adam. ---
def cw_l2(X0, target, c, kappa=0.0, steps=200, lr=0.05):
    X0c = X0.clamp(1e-6, 1-1e-6)
    w = torch.atanh(2*X0c - 1).clone().detach().requires_grad_(True)   # start AT the clean image
    o = torch.optim.Adam([w], lr=lr)                                   # paper's optimizer (Sec V-B)
    for _ in range(steps):
        o.zero_grad()
        xadv = 0.5*(torch.tanh(w) + 1)                                # in [0,1] for ANY w
        logits = Z(xadv)
        tgt_logit = logits.gather(1, target[:,None]).squeeze(1)        # Z_t
        other = logits.clone(); other.scatter_(1, target[:,None], -1e9)
        f = torch.clamp(other.max(1).values - tgt_logit, min=-kappa)   # max(max_{i!=t}Z_i - Z_t, -k)
        loss = ((xadv - X0).pow(2).sum(1) + c*f).sum()                 # ||delta||^2 + c*f
        loss.backward(); o.step()
    return (0.5*(torch.tanh(w)+1)).detach()

# --- 3. FGSM baseline: one signed-gradient step toward the target (the weaker attack). ---
def fgsm(X0, target, eps):
    x = X0.clone().detach().requires_grad_(True)
    g, = torch.autograd.grad(F.cross_entropy(Z(x), target), x)
    return (x - eps*g.sign()).clamp(0,1).detach()

# --- 4. Compare: success rate & mean L2 vs the constant c (C&W) and eps (FGSM). ---
print("\nC&W L2 attack — success & mean L2 vs constant c (kappa=0):")
for c in [0.1, 0.3, 0.5, 1.0, 5.0, 20.0, 100.0]:
    xa = cw_l2(Xc, tgt, c); pred = Z(xa).argmax(1)
    succ = (pred==tgt).float().mean().item()
    l2 = (xa-Xc).pow(2).sum(1).sqrt().mean().item()
    print("  c=%6.1f  success=%.2f  mean L2=%.4f" % (c, succ, l2))

print("\nFGSM baseline — success & mean L2 vs step size eps:")
for eps in [0.1, 0.2, 0.3, 0.4, 0.5]:
    xa = fgsm(Xc, tgt, eps); pred = Z(xa).argmax(1)
    succ = (pred==tgt).float().mean().item()
    l2 = (xa-Xc).pow(2).sum(1).sqrt().mean().item()
    print("  eps=%.2f  success=%.2f  mean L2=%.4f" % (eps, succ, l2))

# worked:  Z=[2.0, 0.5, -1.0] t=2  max_{i!=t}Z=2.0  Z_t=-1.0  f=3.0
# worked2: raw margin=-1.0  f(kappa=1)=-1.0
# classifier train acc = 1.000
# C&W: c=0.3 -> success=1.00 mean L2=0.3253 ;  c=0.5 -> 1.00 / 0.3451
# FGSM: eps=0.40 -> success=0.80 mean L2=0.4767 ; eps=0.50 -> 0.93 / 0.5673
# C&W hits 100% target success at mean L2 ~0.33; FGSM needs far larger L2 for similar success.
# (Our small toy run, not the paper's number. Exact values vary by seed/hardware.)

## Visualize the data & results

_Holding the network fixed, how do success rate and mean L2 distortion trade off — for the C&W L2 attack as we vary the constant c, and for the FGSM baseline as we vary the step size?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
torch.manual_seed(0); np.random.seed(0)

def make_data(n):
    X, y, ctr = [], [], [(-1.5,-1.5),(1.5,-1.5),(0.0,1.6)]
    for c,(cx,cy) in enumerate(ctr):
        X.append(np.random.randn(n,2)*0.45+np.array([cx,cy])); y += [c]*n
    X = np.vstack(X).astype(np.float32); X = (X-X.min(0))/(X.max(0)-X.min(0))
    return torch.tensor(X), torch.tensor(y)

Xtr, ytr = make_data(120)
clf = nn.Sequential(nn.Linear(2,32), nn.ReLU(), nn.Linear(32,32), nn.ReLU(), nn.Linear(32,3))
opt = torch.optim.Adam(clf.parameters(), lr=0.01)
for _ in range(400):
    opt.zero_grad(); F.cross_entropy(clf(Xtr), ytr).backward(); opt.step()
def Z(x): return clf(x)
idx = (clf(Xtr).argmax(1)==ytr).nonzero().squeeze()[:60]
Xc, yc = Xtr[idx].clone(), ytr[idx].clone(); tgt = (yc+1)%3

def cw_l2(X0, target, c, kappa=0.0, steps=200, lr=0.05):
    w = torch.atanh(2*X0.clamp(1e-6,1-1e-6)-1).clone().detach().requires_grad_(True)
    o = torch.optim.Adam([w], lr=lr)
    for _ in range(steps):
        o.zero_grad(); xadv = 0.5*(torch.tanh(w)+1); lg = Z(xadv)
        tl = lg.gather(1, target[:,None]).squeeze(1)
        oth = lg.clone(); oth.scatter_(1, target[:,None], -1e9)
        f = torch.clamp(oth.max(1).values - tl, min=-kappa)
        (((xadv-X0).pow(2).sum(1) + c*f).sum()).backward(); o.step()
    return (0.5*(torch.tanh(w)+1)).detach()

def fgsm(X0, target, eps):
    x = X0.clone().detach().requires_grad_(True)
    g, = torch.autograd.grad(F.cross_entropy(Z(x), target), x)
    return (x - eps*g.sign()).clamp(0,1).detach()

for c in [0.1,0.3,0.5,5.0,20.0,100.0]:
    xa = cw_l2(Xc,tgt,c); p = Z(xa).argmax(1)
    print("CW   c=%6.1f  succ=%.3f  L2=%.4f"
          % (c,(p==tgt).float().mean().item(),(xa-Xc).pow(2).sum(1).sqrt().mean().item()))
for eps in [0.2,0.3,0.4,0.5]:
    xa = fgsm(Xc,tgt,eps); p = Z(xa).argmax(1)
    print("FGSM eps=%.2f  succ=%.3f  L2=%.4f"
          % (eps,(p==tgt).float().mean().item(),(xa-Xc).pow(2).sum(1).sqrt().mean().item()))
# CW   c=   0.1  succ=0.933  L2=0.2896      FGSM eps=0.20 succ=0.117 L2=0.2731
# CW   c=   0.3  succ=1.000  L2=0.3253      FGSM eps=0.30 succ=0.433 L2=0.3837
# CW   c=   0.5  succ=1.000  L2=0.3451      FGSM eps=0.40 succ=0.800 L2=0.4767
# CW   c=   5.0  succ=1.000  L2=0.5276      FGSM eps=0.50 succ=0.933 L2=0.5673
# CW   c=  20.0  succ=1.000  L2=0.5640
# CW   c= 100.0  succ=1.000  L2=0.5727
# C&W: 100% success at L2~0.33; FGSM never beats it at equal distortion.
# Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The softmax-loss ablation. You have a working C&W $L_2$ attack that uses the logit margin
            $f = \max(\max_{i\neq t}Z_i - Z_t, -\kappa)$. Replace it with a softmax cross-entropy loss toward
            the target (so the objective becomes distortion $+\,c\cdot \text{CE}(\text{softmax}(Z), t)$),
            everything else identical. What did you change, and what does the paper predict will happen to the
            search over $c$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Swap the loss: from the logit margin to F.cross_entropy(logits, target), which internally applies softmax. — _Cross-entropy depends on the softmax probabilities, which saturate when the network is confident &mdash; flattening the gradient near a valid image._
- Recall the paper's measurement (Section V-A): the softmax-based gradient is ~$2^{-20}$ near the valid image but ~$2^{-1}$ at the adversarial example &mdash; a ~$10^6$ swing. — _A single constant $c$ cannot balance a term whose gradient changes by six orders of magnitude across the path; the descent is either stuck or unstable._
- Predict: $c$ becomes hard to choose; you need a much larger or finely-tuned $c$, and the attack finds larger perturbations or fails more often than the logit-margin version. — _The logit margin is roughly linear in $Z$, so its gradient is well-scaled everywhere; that is exactly why the paper prefers it._

**Answer:** You replaced the well-scaled logit margin with a softmax cross-entropy loss,
                 reintroducing the saturating-gradient problem the paper avoids. Section V-A reports the
                 softmax-based gradient swings from about $2^{-20}$ near the valid image to $2^{-1}$ at the
                 adversarial point &mdash; roughly a million-fold change &mdash; so no single constant $c$
                 balances distortion against misclassification. Expect a harder, more finicky search over $c$ and
                 generally larger or less reliable perturbations. This is precisely the design reason the C&W
                 attack works on logits, not probabilities.

</details>

**Problem 2.** Why does the C&W $L_2$ attack find smaller $L_2$ perturbations than FGSM at the same success
            level, even though both follow gradients of the same network?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write what FGSM does: one step, $x' = x - \varepsilon\,\text{sign}(\nabla_x \text{loss})$. The same fixed step size $\varepsilon$ is applied to every input and every pixel. — _A fixed-size sign step ignores how far each input actually is from the boundary &mdash; it moves the same amount whether the input is near or far._
- Write what C&W does: minimize $\lVert\delta\rVert_2^2 + c\,f$ over many Adam steps, per input. — _The distortion term actively pulls $\delta$ toward zero, while $f$ only pushes just hard enough to cross the boundary; the optimizer settles at a near-minimal perturbation for each input._
- Tie it to the margin clamp: once $f$ hits $-\kappa$, it stops, so the optimizer spends the rest of its budget shrinking distortion. — _C&W explicitly minimizes distortion subject to crossing the boundary; FGSM has no notion of minimizing distortion at all._

**Answer:** Because C&W optimizes the perturbation while FGSM takes one fixed step. FGSM moves
                 every input by the same $\varepsilon$ in the sign direction, so to fool the stubborn inputs you
                 must raise $\varepsilon$ &mdash; over-perturbing the easy ones and inflating the average $L_2$.
                 C&W minimizes $\lVert\delta\rVert_2^2$ directly, per input, pushing on $f$ only until the
                 target just wins (the $-\kappa$ clamp), then spending the rest of its steps shrinking the
                 change. The result is high success at low distortion. In our run below, C&W reaches 100% target
                 success at mean $L_2 \approx 0.33$, while FGSM at a comparable success spends a substantially
                 larger mean $L_2$.

</details>

**Problem 3.** In the worked example, logits $Z(x')=[2.0, 0.5, -1.0]$, target $t=2$, $\kappa=0$ gave $f=3.0$.
            Suppose the attack progresses and the logits become $Z(x')=[1.0, 0.5, 1.2]$, still $t=2$. Compute
            $f$ with $\kappa=0$, then with $\kappa=0.5$. What does each value tell the optimizer?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Target logit $Z_t = Z_2 = 1.2$. Best competitor among $i\neq t$: $\max(1.0, 0.5) = 1.0$. — _Same recipe: compare the best non-target logit to the target's logit._
- Raw margin $= 1.0 - 1.2 = -0.2$. With $\kappa=0$: $f = \max(-0.2, 0) = 0$. — _The target now barely leads (by $0.2$). With $\kappa=0$ the loss bottoms out at $0$: the attack is satisfied the moment the target wins, and will now only shrink distortion._
- With $\kappa=0.5$: $f = \max(-0.2, -0.5) = -0.2$. — _The target leads by only $0.2$, less than the required margin $0.5$. Since $-0.2 \gt -0.5$, $f$ is not yet saturated &mdash; the loss still has a small negative slope pushing $Z_2$ higher to reach a $0.5$ margin._

**Answer:** With $\kappa=0$: raw margin $1.0-1.2=-0.2$, so $f=\max(-0.2,0)=0$. The target already wins, so
                 the misclassification term is fully satisfied and the optimizer now spends its effort only on
                 shrinking $\lVert\delta\rVert_2^2$. With $\kappa=0.5$: $f=\max(-0.2,-0.5)=-0.2$. Now the
                 target leads by just $0.2$, short of the demanded $0.5$ margin, so $f$ is not at its floor
                 and still pushes the target logit up &mdash; producing a higher-confidence adversarial example at
                 the cost of slightly more distortion. That is exactly what raising $\kappa$ buys you.

</details>